##### Copyright 2019 The TensorFlow Authors.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Step 2: Train a machine learning model

This is the notebook for step 2 of the codelab [**Build a handwritten digit classifier app with TensorFlow Lite**](https://codelabs.developers.google.com/codelabs/digit-classifier-tflite/).

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/examples/blob/master/lite/codelabs/digit_classifier/ml/step2_train_ml_model.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/examples/blob/master/lite/codelabs/digit_classifier/ml/step2_train_ml_model.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

## Import dependencies

We start by importing TensorFlow and other supporting libraries that are used for data processing and visualization.\
我们首先导入 TensorFlow 和其他用于数据处理和可视化的支持库

In [1]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt
import random

print(tf.__version__)

2.18.0


## Download and explore the MNIST dataset

MNIST 数据库包含 60,000 张用于训练的手写数字图像和 10,000 张用于测试的图像。我们将使用该数据集来训练我们的数字分类模型。

MNIST 数据集中的每张图像是 28x28 的灰度图像，包含一个从 0 到 9 的数字，以及一个标识图像中数字的标签。

The MNIST database contains 60,000 training images and 10,000 testing images of handwritten digits. We will use the dataset to train our digit classification model.

Each image in the MNIST dataset is a 28x28 grayscale image containing a digit from 0 to 9, and a label identifying which digit is in the image.
![MNIST sample](https://github.com/khanhlvg/DigitClassifier/raw/master/images/mnist.png)

In [3]:
# Keras provides a handy API to download the MNIST dataset, and split them into
# "train" dataset and "test" dataset.

# Keras 提供了一个便捷的 API 来下载 MNIST 数据集，并将其拆分为
# “训练”数据集和“测试”数据集
mnist = keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Normalize the input image so that each pixel value is between 0 to 1.
# 对输入图像进行归一化，使每个像素值介于 0 到 1 之间
train_images = train_images / 255.0
test_images = test_images / 255.0
print('Pixels are normalized')

In [ ]:
# Show the first 25 images in the training dataset.
# 显示训练数据集中的前 25 张图像。
plt.figure(figsize=(10,10))
for i in range(25):
  plt.subplot(5,5,i+1)
  plt.xticks([])
  plt.yticks([])
  plt.grid(False)
  plt.imshow(train_images[i], cmap=plt.cm.gray)
  plt.xlabel(train_labels[i])
plt.show()

## Train a TensorFlow model to classify digit images

Next, we use Keras API to build a TensorFlow model and train it on the MNIST "train" dataset. After training, our model will be able to classify the digit images.

Our model takes **a 28px x 28px grayscale image** as an input, and outputs **a float array of length 10** representing the probability of the image being a digit from 0 to 9.

Here we use a simple convolutional neural network, which is a common technique in computer vision. We will not go into details about model architecture in this codelab. If you want have a deeper understanding about different ML model architectures, please consider taking our free [TensorFlow training course](https://www.coursera.org/learn/introduction-tensorflow).

## 训练 TensorFlow 模型对数字图像进行分类

接下来，我们使用 Keras API 构建一个 TensorFlow 模型，并在 MNIST 训练数据集上进行训练。训练完成后，我们的模型将能够对数字图像进行分类。

我们的模型以**28px x 28px 的灰度图像**作为输入，并输出**一个长度为 10 的浮点数组**，该数组表示图像为 0 到 9 的数字的概率。

这里我们使用一个简单的卷积神经网络，这是计算机视觉中的一种常用技术。在本 Codelab 中，我们不会详细介绍模型架构。如果您想深入了解不同的机器学习模型架构，请考虑参加我们的免费 [TensorFlow 培训课程](https://www.coursera.org/learn/introduction-tensorflow)。

In [ ]:
# Define the model architecture
# 定义模型架构
model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation=tf.nn.relu),
  keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation=tf.nn.relu),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Dropout(0.25),
  keras.layers.Flatten(),
  keras.layers.Dense(10)
])

# Define how to train the model
# 定义如何训练模型
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train the digit classification model
# 训练数字分类模型
model.fit(train_images, train_labels, epochs=5)

Let's take a closer look at our model structure.
让我们仔细看看我们的模型结构。

In [ ]:
model.summary()

There is an extra dimension with **None** shape in every layer in our model,

---

which is called the **batch dimension**. In machine learning, we usually process data in batches to improve throughput, so TensorFlow automatically add the dimension for you.

我们的模型中，每一层都有一个形状为**None**的额外维度，

---

这被称为**批次维度**。在机器学习中，我们通常以批次形式处理数据以提高吞吐量，因此 TensorFlow 会自动为您添加此维度。

## Evaluate our model
We run our digit classification model against our "test" dataset that the model has not seen during its training process to confirm that the model did not just remember the digits it saw but also generalize well to new images.

## 评估我们的模型
我们针对模型在训练过程中未曾见过的“测试”数据集运行数字分类模型，以确认该模型不仅能够记住它所见过的数字，而且还能很好地推广到新图像。

In [ ]:
# Evaluate the model using all images in the test dataset.
# 使用测试数据集中的所有图像评估模型。
test_loss, test_acc = model.evaluate(test_images, test_labels)

print('Test accuracy:', test_acc)

Although our model is relatively simple, we were able to achieve good accuracy around 98% on new images that the model has never seen before. Let's visualize the result.

虽然我们的模型相对简单，但我们能够在模型从未见过的新图像上达到约 98% 的准确率。让我们将结果可视化。

In [ ]:
# A helper function that returns 'red'/'black' depending on if its two input
# parameter matches or not.
# # 一个辅助函数，根据两个输入参数是否匹配，返回“红色”/“黑色”。
def get_label_color(val1, val2):
  if val1 == val2:
    return 'black'
  else:
    return 'red'

# Predict the labels of digit images in our test dataset.
# 预测我们测试数据集中数字图像的标签。
predictions = model.predict(test_images)

# As the model output 10 float representing the probability of the input image
# being a digit from 0 to 9, we need to find the largest probability value
# to find out which digit the model predicts to be most likely in the image.

# 由于模型输出 10 个浮点数，表示输入图像中数字的概率，
# 因此我们需要找到最大的概率值，
# 从而找出模型预测图像中最有可能出现的数字。
prediction_digits = np.argmax(predictions, axis=1)

# Then plot 100 random test images and their predicted labels.
# If a prediction result is different from the label provided label in "test"
# dataset, we will highlight it in red color.
# 然后绘制 100 张随机测试图像及其预测标签。
# 如果预测结果与“测试”数据集中提供的标签不同，我们将用红色突出显示。
plt.figure(figsize=(18, 18))
for i in range(100):
  ax = plt.subplot(10, 10, i+1)
  plt.xticks([])
  plt.yticks([])
  plt.grid(False)
  image_index = random.randint(0, len(prediction_digits))
  plt.imshow(test_images[image_index], cmap=plt.cm.gray)
  ax.xaxis.label.set_color(get_label_color(prediction_digits[image_index],\
                                           test_labels[image_index]))
  plt.xlabel('Predicted: %d' % prediction_digits[image_index])
plt.show()

## Convert the Keras model to TensorFlow Lite
## 将 Keras 模型转换为 TensorFlow Lite

Now as we have trained the digit classifer model, we will convert it to TensorFlow Lite format for mobile deployment.

现在我们已经训练了数字分类器模型，我们将把它转换为 TensorFlow Lite 格式以进行移动部署。

In [ ]:
# Convert Keras model to TF Lite format.
# 将 Keras 模型转换为 TF Lite 格式。

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float_model = converter.convert()

# Show model size in KBs.
# 以 KB 为单位显示模型大小。
float_model_size = len(tflite_float_model) / 1024
print('Float model size = %dKBs.' % float_model_size)

As we will deploy our model to a mobile device, we want our model to be as small and as fast as possible. **Quantization** is a common technique often used in on-device machine learning to shrink ML models. Here we will use 8-bit number to approximate our 32-bit weights, which in turn shrinks the model size by a factor of 4.

See [TensorFlow documentation](https://www.tensorflow.org/lite/performance/post_training_quantization) to learn more about other quantization techniques.


由于我们将模型部署到移动设备上，因此我们希望模型尽可能小巧、运行速度尽可能快。**量化**是设备端机器学习中常用的一种技术，用于缩小机器学习模型的大小。在这里，我们将使用 8 位数字来近似 32 位权重，从而将模型大小缩小 4 倍。

请参阅 [TensorFlow 文档](https://www.tensorflow.org/lite/performance/post_training_quantization)，了解更多关于其他量化技术的信息。

In [ ]:
# Re-convert the model to TF Lite using quantization.
# # 使用量化将模型重新转换为 TF Lite。
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

# Show model size in KBs.
# 以 KB 为单位显示模型大小。
quantized_model_size = len(tflite_quantized_model) / 1024
print('Quantized model size = %dKBs,' % quantized_model_size)
print('which is about %d%% of the float model size.'\
      % (quantized_model_size * 100 / float_model_size))

## Evaluate the TensorFlow Lite model

By using quantization, we often traded off a bit of accuracy for the benefit of having a significantly smaller model. Let's calculate the accuracy drop of our quantized model.

## 评估 TensorFlow Lite 模型

通过量化，我们通常会牺牲一些准确率，以换取显著减小的模型体积。让我们计算一下量化模型的准确率下降情况。

In [ ]:
# 一个辅助函数，用于使用“测试”数据集评估TF Lite模型。
def evaluate_tflite_model(tflite_model):
  # 使用模型初始化TFLite解释器。
  interpreter = tf.lite.Interpreter(model_content=tflite_model)
  interpreter.allocate_tensors()
  input_tensor_index = interpreter.get_input_details()[0]["index"]
  output = interpreter.tensor(interpreter.get_output_details()[0]["index"])

  # 对“测试”数据集中的每张图像进行预测。
  prediction_digits = []
  for test_image in test_images:
    # 预处理：添加批次维度并转换为float32，以匹配模型的输入数据格式。
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_tensor_index, test_image)

    # 运行推理。
    interpreter.invoke()

    # 后处理：移除批次维度并找到概率最高的数字。
    digit = np.argmax(output()[0])
    prediction_digits.append(digit)

  # 将预测结果与真实标签比较，计算精度。
  accurate_count = 0
  for index in range(len(prediction_digits)):
    if prediction_digits[index] == test_labels[index]:
      accurate_count += 1
  accuracy = accurate_count * 1.0 / len(prediction_digits)

  return accuracy

# 评估TF Lite浮点模型。你会发现其精度与原始TF（Keras）模型相同，因为它们本质上是相同模型，只是存储格式不同。
float_accuracy = evaluate_tflite_model(tflite_float_model)
print('浮点模型精度 = %.4f' % float_accuracy)

# 评估TF Lite量化模型。
# 如果量化模型的精度高于原始浮点模型，不要感到惊讶，这种情况有时会发生 :)
quantized_accuracy = evaluate_tflite_model(tflite_quantized_model)
print('量化模型精度 = %.4f' % quantized_accuracy)
print('精度下降 = %.4f' % (float_accuracy - quantized_accuracy))

## Download the TensorFlow Lite model

Let's get our model and integrate it into an Android app.

If you see an error when downloading mnist.tflite from Colab, try running this cell again.

## 下载 TensorFlow Lite 模型

让我们获取模型并将其集成到 Android 应用中。

如果您在从 Colab 下载 mnist.tflite 时遇到错误，请尝试再次运行此单元。

In [ ]:
# 将量化模型保存到Downloads目录下的文件中
f = open('mnist.tflite', "wb")
f.write(tflite_quantized_model)
f.close()

# 下载数字分类模型
from google.colab import files
files.download('mnist.tflite')

print('`mnist.tflite` 已下载')

## Good job!
This is the end of *Step 2: Train a machine learning model* in the codelab **Build a handwritten digit classifier app with TensorFlow Lite**. Let's go back to our codelab and proceed to the [next step](https://codelabs.developers.google.com/codelabs/digit-classifier-tflite/#2).

## 干得好！
Codelab 中的“*步骤 2：训练机器学习模型*”**使用 TensorFlow Lite 构建手写数字分类器应用**已结束。让我们回到 Codelab 并继续[下一步](https://codelabs.developers.google.com/codelabs/digit-classifier-tflite/#2)。